# MNIST: model comparison & Explainable AI

The new workflow uses the `mnist_project` package, shares one train/validation split across models, and keeps the official test set separate. By default it runs a small CNN experiment for one epoch to check the workflow. See the README for full-data and Optuna commands. Results here are different from the historical numbers in `reports/notebook/`.

Install from the project directory with `python -m pip install -r requirements.txt`. Select a kernel from the installed environment and run cells in order.


In [ ]:
from pathlib import Path
import os
import json
from datetime import datetime, timezone
import numpy as np
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks directory.")
os.chdir(ROOT)

from mnist_project.data import load_data
from mnist_project.eda import create_eda
from mnist_project.experiments import run_experiment, predict_images
from mnist_project.explain import explain_run

QUICK = True
# To run offline, set MNIST_DATA to the path of a Keras-style MNIST NPZ.
DATA_PATH = os.environ.get("MNIST_DATA") or None
RUN = ROOT / "artifacts" / ("notebook-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
print("Output:", RUN)


## 1. Data

By default, split 60,000 training images into 54,000 train and 6,000 validation images; keep the 10,000 official test images separate. QUICK takes stratified subsets of training and test data while leaving validation intact. The first run needs Internet access to download MNIST if it is not cached.


In [ ]:
data = load_data(DATA_PATH, seed=42, train_limit=2000 if QUICK else None,
                 test_limit=500 if QUICK else None)
print({"train": len(data.y_train), "validation": len(data.y_val), "test": len(data.y_test)})
assert not set(data.train_indices) & set(data.val_indices)
assert data.x_train.dtype == np.float32


## 2. EDA

Save the class distribution, digit examples, mean images, pixel histogram, and sparsity plots.


In [ ]:
summary = create_eda(data, RUN / "eda")
display(Image(filename=str(RUN / "eda" / "samples_and_means.png")))


## 3. Training and evaluation

Run a CNN as an example. Change MODEL to `knn`, `svm`, `random_forest`, `mlp`, `resnet9`, or `tinyvit` for another model. For a seven-model benchmark, use `python -m mnist_project train --model all --output artifacts/full-run`. QUICK and synthetic data only check the workflow; they do not measure model quality.


In [ ]:
MODEL = "cnn"
model_dir = RUN / MODEL
metrics = run_experiment(MODEL, data, model_dir, epochs=1 if QUICK else 15, batch_size=64)
{k: metrics[k] for k in ["accuracy", "f1_macro", "f1_weighted", "n_test", "train_seconds"]}


In [ ]:
display(Image(filename=str(model_dir / "confusion_matrix.png")))
if (model_dir / "learning_curves.png").exists():
    display(Image(filename=str(model_dir / "learning_curves.png")))

## 4. Saving, loading, and prediction

The runner saved the model and metadata. `predict_images` reloads the model from disk. SVM returns `decision_scores`, not probabilities. Do not load models from untrusted sources.


In [ ]:
result = predict_images(model_dir, data.x_test[:5])
print("True:", data.y_test[:5].tolist())
print("Pred:", result["predictions"])


## 5. Explanations

`auto` selects a suitable method for MODEL: Grad-CAM for CNN/ResNet, attention rollout for Tiny-ViT, and IG for MLP. Use `method="ig"` for any deep model to inspect signed attributions and a single-image deletion test. An undertrained model may produce a weak or empty map; a map does not prove correctness.


In [ ]:
explanation = explain_run(model_dir, data.x_test[:1], RUN / "explanation", method="auto")
display(Image(filename=str(RUN / "explanation" / "explanation.png")))
explanation


## 6. Optuna (optional; does not run automatically)

Tune on training and validation data only. Keep the same seed, split, and subset when training from `best_params`. Do not use the official test set to choose hyperparameters.

```python
from mnist_project.tuning import tune
best = tune("cnn", data, RUN / "search", trials=5, epochs=10)
print(best["best_params"])
```

Fresh results are saved in `artifacts/`; historical results are in `reports/notebook/`. Read `docs/methodology.md` before comparing the two.
